### Overview
This notebook demonstrates an end-to-end machine learning pipeline for multi-class cancer classification using RNA-Seq gene expression data.  
The workflow includes data loading, exploratory analysis, preprocessing, baseline model training, and evaluation.  
The goal is to build a strong baseline that can be extended with advanced models and feature selection.


In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression


In [2]:
import os
os.listdir("/kaggle/input")


['rna-seq-gene-expression-datase-multi-class-cancer']

In [3]:
import pandas as pd

train_df = pd.read_csv(
    "/kaggle/input/rna-seq-gene-expression-datase-multi-class-cancer/train.csv"
)

test_df = pd.read_csv(
    "/kaggle/input/rna-seq-gene-expression-datase-multi-class-cancer/test.csv"
)


In [4]:
train_df.head()


,Id,gene_1,gene_3,gene_5,gene_7,gene_8,gene_9,gene_10,gene_11,gene_13,...,gene_20634,gene_20635,gene_20636,gene_20637,gene_20638,gene_20639,gene_20640,gene_20641,gene_20642,Class
0,sample_664,0.160738,-0.327348,-0.144638,0.196493,-1.105093,0.309926,-0.177461,-1.124182,-0.459826,...,-1.611378,-1.108411,-0.670719,-1.739299,0.476467,1.136071,-0.576601,-1.275518,-0.508678,1.0
1,sample_215,-0.771173,0.885819,-0.234209,0.273139,0.132208,-0.249541,0.005817,-0.631647,NaN,...,0.247812,0.144035,0.148776,-1.373208,0.099245,0.391993,0.573363,0.322198,6.022439,0.0
2,sample_343,-0.169258,1.908618,0.165008,-0.562826,0.199720,0.128036,2.348450,2.425346,-0.933545,...,1.133065,0.965014,1.873753,-0.005167,-0.223091,0.782868,-0.562787,-0.471593,-0.763284,4.0
3,sample_707,-0.947912,0.111177,-0.153179,0.837412,0.185467,-0.066223,-0.267734,0.674365,-0.076086,...,0.022339,0.326506,-0.333964,0.228595,-0.245309,0.478564,0.273364,1.756369,-0.266200,1.0
4,sample_621,-0.335741,0.515251,0.325440,-0.842387,-0.500415,0.484240,-0.438587,-0.874562,NaN,...,-1.516812,-1.430622,-0.664933,-0.753410,NaN,0.375521,-0.536705,-0.523850,0.222560,4.0


In [5]:
train_df.columns


Index(['Id', 'gene_1', 'gene_3', 'gene_5', 'gene_7', 'gene_8', 'gene_9',
       'gene_10', 'gene_11', 'gene_13',
       ...
       'gene_20634', 'gene_20635', 'gene_20636', 'gene_20637', 'gene_20638',
       'gene_20639', 'gene_20640', 'gene_20641', 'gene_20642', 'Class'],
      dtype='object', length=14574)

## Dataset Overview

This dataset consists of high-dimensional RNA-Seq gene expression data.
Each row represents a biological sample, and each column represents a gene expression feature.
The goal is to predict the cancer class using supervised machine learning.

- **Id**: Unique sample identifier (not used for modeling)
- **gene_***: Normalized gene expression features
- **Class**: Target label representing cancer type (multi-class)


In [6]:
# Separate features and target
X = train_df.drop(columns=["Id", "Class"])
y = train_df["Class"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


Feature matrix shape: (400, 14572)
Target shape: (400,)


## Target Distribution

Understanding class balance is important for evaluating model performance
and deciding whether advanced techniques such as class weighting are needed.


In [7]:
y.value_counts()


Class
2.0    56
4.0    27
1.0    26
0.0    26
3.0    15
Name: count, dtype: int64

## Train–Validation Split

The dataset is split into training and validation sets to evaluate model
performance on unseen data.




## Handling Missing Target Labels

Before splitting the data, it is necessary to ensure that the target variable
does not contain missing values. Rows with missing class labels are removed
to enable proper stratified sampling.


In [8]:
train_df["Class"].isna().sum()


np.int64(250)

In [9]:
# Remove rows where target is missing
train_df = train_df.dropna(subset=["Class"])

# Reset index after removal
train_df = train_df.reset_index(drop=True)

print("Remaining samples:", train_df.shape[0])


Remaining samples: 150


In [10]:
X = train_df.drop(columns=["Id", "Class"])
y = train_df["Class"]


## Train–Validation Split

After cleaning the dataset, a stratified train–validation split is performed
to preserve class distribution across splits.


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_val.shape


((120, 14572), (30, 14572))

## Feature Scaling

RNA-Seq gene expression data is high-dimensional and varies in scale.
Standardization is applied to ensure stable and efficient model training.


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

X_train_scaled.shape, X_val_scaled.shape


((120, 14572), (30, 14572))

## Baseline Model: Logistic Regression

Logistic Regression is used as a baseline model for multi-class cancer
classification to establish initial performance.


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, n_jobs=-1))
])


In [14]:
pipeline.fit(X_train, y_train)


Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000, n_jobs=-1))])

## Model Evaluation

The pipeline is evaluated on the validation set to measure
classification performance after handling missing values.


In [15]:
from sklearn.metrics import accuracy_score, classification_report

val_preds = pipeline.predict(X_val)

print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print(classification_report(y_val, val_preds))


Validation Accuracy: 1.0
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00         5
         1.0       1.00      1.00      1.00         5
         2.0       1.00      1.00      1.00        11
         3.0       1.00      1.00      1.00         3
         4.0       1.00      1.00      1.00         6

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



## Generating Predictions and Submission File

After training the machine learning pipeline, predictions are generated
for the test dataset and formatted according to Kaggle’s submission
requirements.


In [16]:
# Separate features from Id in test data
X_test = test_df.drop(columns=["Id"])

print("Test shape:", X_test.shape)


Test shape: (401, 14572)


In [17]:
# Predict using trained pipeline
test_preds = pipeline.predict(X_test)

test_preds[:5]


array([2., 0., 3., 1., 0.])

In [18]:
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "Class": test_preds
})

submission.head()


,Id,Class
0,sample_313,2.0
1,sample_427,0.0
2,sample_573,3.0
3,sample_214,1.0
4,sample_793,0.0


In [19]:
submission.to_csv("submission.csv", index=False)


## Conclusion

An end-to-end machine learning pipeline was built to classify cancer types
using high-dimensional RNA-Seq gene expression data. The workflow included
data cleaning, handling missing values, feature scaling, model training,
and prediction generation. This baseline model can be further improved
using dimensionality reduction or advanced ensemble methods.
